# P1 — Horizon Discovery: which (feature, horizon) pairs carry signal?

**Slug**: `crypto_intraday/10_horizon_discovery`

**Status**: in_progress

**Research question**: Before committing to a working interval / horizon for the rule-based and ML phases, empirically determine which lookback × horizon combinations carry information content for BTCUSDT / ETHUSDT perp 1m. The output drives P1-4 (rule-based) and P3 (ML) scope.

**Method**:
1. Resample 1m → 5m (the workhorse interval for first pass).
2. Build a small catalog of leak-safe features at different lookbacks.
3. For each (feature, horizon ∈ {5m, 15m, 30m, 1h, 4h}) pair, compute IC and rank-IC vs the safe forward return.
4. Block-bootstrap CIs (block ~ 1 day) to size the noise floor.
5. Decision: keep (feature, horizon) pairs whose IC magnitude is at least the noise-floor bar AND whose 95% CI excludes 0.

**Window**: Q1+Q2 2024 of train data (well before validation). Hold-out is enforced by `safe_forward_returns`.

In [ ]:
from itertools import product

import numpy as np
import pandas as pd

from alpha_lab.data.holdout import PMHoldout, access_summary_for_report, safe_forward_returns
from alpha_lab.data.loaders import binance_vision as bv
from alpha_lab.features import intraday as f
from alpha_lab.ml.cv import BlockBootstrap
from alpha_lab.utils.config import load_config

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 200)

In [ ]:
cfg = load_config('crypto_intraday')
holdout = PMHoldout.from_config('crypto_intraday')

SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MARKET = 'perp'
ARCHIVE_INTERVAL = '1m'
RES_START = '2024-01-01'
RES_END = '2024-07-01'   # half a year of train

# Workhorse interval for this notebook
WORK_INTERVAL = '5min'   # pandas offset
WORK_INTERVAL_MIN = 5
BARS_PER_DAY_5M = 24 * 60 // WORK_INTERVAL_MIN  # 288
print(f'Window: [{RES_START}, {RES_END}) — {pd.Timestamp(RES_END) - pd.Timestamp(RES_START)}')

## 1. Load 1m and resample to 5m

In [ ]:
panel_1m = bv.load_klines(
    SYMBOLS, ARCHIVE_INTERVAL,
    start=RES_START, end=RES_END, market=MARKET, holdout=holdout,
)
print('1m panel sizes:')
for sym, df in panel_1m.frames.items():
    print(f'  {sym}: {len(df):>7d} bars  [{df.index.min()} → {df.index.max()}]')

In [ ]:
# Right-labeled OHLCV resample to 5min
def resample_ohlcv(df: pd.DataFrame, offset: str) -> pd.DataFrame:
    agg = {
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
        'volume': 'sum', 'quote_volume': 'sum', 'n_trades': 'sum',
        'taker_buy_base': 'sum',
    }
    return df.resample(offset, label='right', closed='right').agg(agg).dropna(how='all')

ohlcv_5m = {sym: resample_ohlcv(df, WORK_INTERVAL) for sym, df in panel_1m.frames.items()}
for sym, df in ohlcv_5m.items():
    print(f'5m {sym}: {len(df):>6d} bars')

## 2. Feature catalog

Small set of simple, leak-safe features. Each is a per-symbol Series.

In [ ]:
def build_features(o: pd.DataFrame) -> dict[str, pd.Series]:
    return {
        'ret_1bar':       f.log_return(o['close'], window=1),     # last 5m
        'ret_6bar':       f.log_return(o['close'], window=6),     # last 30m
        'ret_12bar':      f.log_return(o['close'], window=12),    # last 1h
        'ret_48bar':      f.log_return(o['close'], window=48),    # last 4h
        'rvol_60bar':     f.realized_vol_close(o['close'], window=60),
        'vol_z_60bar':    f.volume_zscore(o['volume'], window=60),
        'distma_60bar':   f.distance_from_ma(o['close'], window=60),
        'rsi_14bar':      f.rsi(o['close'], window=14),
        'bollb_20bar':    f.bollinger_pct_b(o['close'], window=20),
    }

features = {sym: build_features(df) for sym, df in ohlcv_5m.items()}
feature_names = list(next(iter(features.values())).keys())
print(f'{len(feature_names)} features × {len(SYMBOLS)} symbols')
print(feature_names)

## 3. Forward returns at horizons {1, 3, 6, 12, 48} 5m-bars = {5m, 15m, 30m, 1h, 4h}

`safe_forward_returns` masks any label whose forward window peeks into the PM holdout. For Q1+Q2 2024 that's a no-op (holdout is in 2026), but the call is the right discipline pattern.

In [ ]:
HORIZONS_BARS = [1, 3, 6, 12, 48]
HORIZON_LABELS = {1: '5m', 3: '15m', 6: '30m', 12: '1h', 48: '4h'}

fwd_returns: dict[tuple[str, int], pd.Series] = {}
for sym, o in ohlcv_5m.items():
    rets = o['close'].pct_change().fillna(0.0)
    for h in HORIZONS_BARS:
        fwd = safe_forward_returns(rets, horizon=h, holdout=holdout)
        fwd_returns[(sym, h)] = fwd
print(f'{len(fwd_returns)} (symbol, horizon) forward-return series')

## 4. IC + rank-IC + bootstrap CI

In [ ]:
def _aligned(a: pd.Series, b: pd.Series) -> tuple[pd.Series, pd.Series]:
    df = pd.concat([a, b], axis=1).dropna()
    return df.iloc[:, 0], df.iloc[:, 1]

def ic(a: pd.Series, b: pd.Series) -> float:
    aa, bb = _aligned(a, b)
    if len(aa) < 30:
        return float('nan')
    return float(aa.corr(bb))

def rank_ic(a: pd.Series, b: pd.Series) -> float:
    aa, bb = _aligned(a, b)
    if len(aa) < 30:
        return float('nan')
    return float(aa.corr(bb, method='spearman'))

def bootstrap_ic_ci(
    feature: pd.Series, fwd: pd.Series,
    block_bars: int, n_resamples: int = 300, seed: int = 0,
) -> tuple[float, float]:
    """Block-bootstrap 95% CI on Pearson IC. block_bars in 5m bars."""
    aligned = pd.concat([feature, fwd], axis=1).dropna()
    if len(aligned) < 5 * block_bars:
        return float('nan'), float('nan')
    bb = BlockBootstrap(block_size=block_bars, n_resamples=n_resamples, seed=seed,
                        mode='stationary')
    samples = []
    for resampled_idx in bb.resample(aligned.index):
        sub = aligned.loc[resampled_idx]
        # When resamples include duplicates (block bootstrap does), pandas .corr
        # still works element-wise on the duplicated rows.
        c = sub.iloc[:, 0].corr(sub.iloc[:, 1])
        if not np.isnan(c):
            samples.append(c)
    if not samples:
        return float('nan'), float('nan')
    arr = np.array(samples)
    return float(np.quantile(arr, 0.025)), float(np.quantile(arr, 0.975))

In [ ]:
rows: list[dict] = []
BLOCK_BARS = BARS_PER_DAY_5M  # 1-day block ~= 288 bars

for sym, h in product(SYMBOLS, HORIZONS_BARS):
    fwd = fwd_returns[(sym, h)]
    for feat_name, feat_series in features[sym].items():
        ic_val = ic(feat_series, fwd)
        rank_ic_val = rank_ic(feat_series, fwd)
        ci_lo, ci_hi = bootstrap_ic_ci(
            feat_series, fwd, block_bars=BLOCK_BARS, n_resamples=200, seed=0,
        )
        rows.append({
            'symbol': sym, 'feature': feat_name,
            'horizon_bars': h, 'horizon': HORIZON_LABELS[h],
            'IC': ic_val, 'rank_IC': rank_ic_val,
            'IC_CI_lo': ci_lo, 'IC_CI_hi': ci_hi,
            'CI_excludes_zero': bool(ci_lo > 0 or ci_hi < 0),
        })
results = pd.DataFrame(rows)
results.head(15)

## 5. Heatmaps (IC by feature × horizon, per symbol)

In [ ]:
for sym in SYMBOLS:
    print(f'\n=== {sym} — IC (Pearson) ===')
    pivot = results[results['symbol'] == sym].pivot(
        index='feature', columns='horizon', values='IC',
    )
    pivot = pivot.reindex(columns=['5m', '15m', '30m', '1h', '4h'])
    pivot = pivot.reindex(index=feature_names)
    print(pivot)

In [ ]:
for sym in SYMBOLS:
    print(f'\n=== {sym} — rank IC (Spearman) ===')
    pivot = results[results['symbol'] == sym].pivot(
        index='feature', columns='horizon', values='rank_IC',
    )
    pivot = pivot.reindex(columns=['5m', '15m', '30m', '1h', '4h'])
    pivot = pivot.reindex(index=feature_names)
    print(pivot)

## 6. Signal-survival pass

Keep (feature, horizon, symbol) tuples where:
- |IC| ≥ 0.01 (a modest minimum that filters obvious noise)
- the 95% bootstrap CI excludes 0 (the effect is statistically distinguishable from zero at this sample size).

These are NOT alpha claims — IC ~ 1% before costs is barely-survivable. They are pointers for the rule-based and ML notebooks: focus there, ignore the rest.

In [ ]:
alive = results[
    (results['IC'].abs() >= 0.01) & (results['CI_excludes_zero'])
].sort_values('IC', key=lambda s: s.abs(), ascending=False)
print(f'Surviving (feature, horizon, symbol): {len(alive)} of {len(results)}')
print()
print(alive[['symbol', 'feature', 'horizon', 'IC', 'rank_IC', 'IC_CI_lo', 'IC_CI_hi']]
      .to_string(index=False))

In [ ]:
# Which horizons appear most often in the surviving set?
print('Surviving counts by horizon:')
print(alive['horizon'].value_counts())
print()
print('Surviving counts by feature:')
print(alive['feature'].value_counts())

## 7. PM holdout audit

In [ ]:
audit = access_summary_for_report()
print('PM Holdout audit:')
for k, v in audit.items():
    print(f'  {k}: {v}')
if audit['accessed']:
    raise RuntimeError('PM HOLDOUT WAS ACCESSED — contaminated.')
print('\nPM Holdout was not accessed.')

## Decision

Decision is recorded in `docs/research_decisions/crypto_intraday/P1-horizon-discovery.md` after this notebook runs. The conclusions encoded there drive the choice of (interval, horizon) used by `11_rule_based.ipynb`.

**Mechanical conclusion drawn here** (re-derived each run):
- The set of surviving (feature, horizon, symbol) tuples printed above defines the working scope.
- If multiple horizons survive, prefer the LONGEST surviving horizon for cost reasons (lower turnover for a given signal change rate).
- If the surviving set is empty, the rule-based phase reverts to the most-traded literature defaults (e.g. RSI(14) at 5m, MA(60) trend filter at 5m) and explicitly notes that no edge was empirically established at this stage.